# Econ 5200 PS3 Warm-up — the code PS3 asks you to write

**Not graded.** Practice for Problem Set 3, on different data. Each exercise
shows you working code (*Watch*), asks you to write a similar piece yourself
(*Your turn*), then checks it. The exercises follow PS3's order: a two-period table made long, a nested panel with the truth on file, TWFE and one forbidden comparison, one clean comparison, an event study, then three standard errors.

- **Turn AI code completion off** (Colab: *Settings › AI assistance*), as PS3
  Phases 1–3 require. You may still click **Explain error**.
- The *Watch* cells use stores in sales regions that adopt a loyalty app and the *Your turn* cells use schools in districts that start a tutoring programme. The two
  differ **on purpose**: you write each piece yourself instead of copying it.
- Run the cells in order: later exercises use what earlier ones made.
- If a check fails, read its message, fix your cell, and run both again.
- Stuck for more than five minutes? Look back at the *Watch* cell above it.
- Phase 4's wild cluster bootstrap is AI work and is not practised here.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

---
## 1. A two-period table made long, and the 2×2 as a coefficient (PS3 Step 1.2)

A survey that visits each unit twice usually arrives **wide**: one row per unit, one column per visit. A regression wants it **long**: one row per unit and visit. `df.melt(id_vars=[...], value_vars=[...], var_name=..., value_name=...)` stacks the visit columns, and a 0/1 `post` column says which visit a row is. A missed visit becomes a row with a missing outcome: drop it, do not fill it with zero. In `smf.ols("y ~ treated * post")` the coefficient named `treated:post` is the difference of the two changes, the same number as the 2×2 by hand. The two main effects are group means too: read off which.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(4)
n_stores = 300
app_region = (rng.random(n_stores) < 0.5).astype(int)     # 1 for stores in the region that launched the app
level = rng.normal(100, 15, n_stores)                     # each store's own sales level
stores_wide = pd.DataFrame({
    "store": np.arange(n_stores),
    "app_region": app_region,
    "sales_before": level + 5 * app_region + rng.normal(0, 8, n_stores),
    "sales_after": level + 5 * app_region - 3 + 4 * app_region + rng.normal(0, 8, n_stores)})
missed = rng.choice(n_stores, size=12, replace=False)
stores_wide.loc[missed, "sales_after"] = np.nan           # 12 stores were not visited the second time

# By hand: four means, two changes, one difference
m = stores_wide.groupby("app_region")[["sales_before", "sales_after"]].mean()
did_hand = ((m.loc[1, "sales_after"] - m.loc[1, "sales_before"])
            - (m.loc[0, "sales_after"] - m.loc[0, "sales_before"]))

# Long: one row per store and visit; post is 1 for the second visit
stores_long = stores_wide.melt(id_vars=["store", "app_region"],
                               value_vars=["sales_before", "sales_after"],
                               var_name="visit", value_name="sales")
stores_long["post"] = (stores_long["visit"] == "sales_after").astype(int)
stores_long = stores_long.dropna(subset=["sales"])        # a missed visit has no outcome to fit
fit_stores = smf.ols("sales ~ app_region * post", data=stores_long).fit()

print(fit_stores.params.round(3))
print(f"2x2 by hand {did_hand:.3f} | interaction {fit_stores.params['app_region:post']:.3f}")
print(f"app_region = the gap at the first visit: {m.loc[1, 'sales_before'] - m.loc[0, 'sales_before']:.3f}")
print(f"post = the change in the other region: {m.loc[0, 'sales_after'] - m.loc[0, 'sales_before']:.3f}")

**Your turn:** the schools below are wide, with a fall and a spring reading score, and 10 schools missed the spring test. Compute the 2×2 by hand from the wide table as `did_by_hand`. Then reshape to `scores_long` (columns `school`, `tutored`, `post`, `score`; `post` is 1 for spring; no missing scores) and fit `fit_scores = smf.ols("score ~ tutored * post", ...)`.

In [ ]:
rng = np.random.default_rng(8)
n_schools = 240
tutored = (rng.random(n_schools) < 0.4).astype(int)       # 1 for schools in the district that started tutoring
level = rng.normal(60, 10, n_schools)
scores_wide = pd.DataFrame({
    "school": np.arange(n_schools),
    "tutored": tutored,
    "score_fall": level - 4 * tutored + rng.normal(0, 5, n_schools),
    "score_spring": level - 4 * tutored + 2 + 3 * tutored + rng.normal(0, 5, n_schools)})
scores_wide.loc[rng.choice(n_schools, size=10, replace=False), "score_spring"] = np.nan

# YOUR CODE HERE
did_by_hand = scores_long = fit_scores = None

In [ ]:
assert did_by_hand is not None, "did_by_hand: the change for tutored schools minus the change for the others"
assert scores_long is not None and {"school", "tutored", "post", "score"} <= set(scores_long.columns), \
    "scores_long needs the columns school, tutored, post and score"
assert len(scores_long) == 470 and scores_long["score"].notna().all(), \
    "240 fall rows plus 230 spring rows: drop the 10 missing spring scores"
assert set(scores_long["post"]) == {0, 1} and (scores_long["post"] == 1).sum() == 230, "post is 1 for spring rows only"
assert fit_scores is not None and "tutored:post" in fit_scores.params, "fit score ~ tutored * post on scores_long"
assert abs(fit_scores.params["tutored:post"] - did_by_hand) < 1e-8, \
    "the interaction should equal the 2x2 by hand to every digit"
gap_fall = scores_wide.groupby("tutored")["score_fall"].mean()
assert abs(fit_scores.params["tutored"] - (gap_fall[1] - gap_fall[0])) < 1e-8, "tutored should be the fall gap"
print(f"interaction {fit_scores.params['tutored:post']:.3f} = 2x2 by hand {did_by_hand:.3f}")
print(f"tutored {fit_scores.params['tutored']:.3f} | post {fit_scores.params['post']:.3f}")
print("Which group mean, or change in means, is each of those two? Why is neither one the effect of tutoring?")

---
## 2. A nested panel with the truth on file (PS3 Step 2.1)

Policies are often switched on for a whole **group** at once, so the simulation nests units in groups. Loop over groups, then the units in a group, then the periods. Draw what the group shares **outside** the unit loop: its adoption period `g` and one shock per period (`rng.normal(0, sd, n_periods)`), so every unit in the group gets the same `shock[t]`. Draw what is the unit's own, its level, inside. An effect that grows with exposure is `growth * (t - g + 1)` from `t = g` on, and 0 before. Store it in its own column: that is the truth you will grade every estimator against. A group that never adopts gets `g = NEVER`, a number later than any period, as in Lab 11.

**Watch:** run this and read it.

In [ ]:
rng = np.random.default_rng(21)
NEVER = 10**9                                   # the start period of a region that never adopts
region_start = {0: 3, 1: 3, 2: 5, 3: 5, 4: 7, 5: NEVER}   # first quarter with the app, by region
STORES_PER_REGION, QUARTERS = 6, 10
GROWTH_APP = 0.8                                # extra sales per quarter of exposure

rows = []
for region, g in region_start.items():
    shock = rng.normal(0, 0.3, QUARTERS)        # one draw per region and quarter, shared by its stores
    for k in range(STORES_PER_REGION):
        store = f"r{region}s{k}"
        own = rng.normal(0, 2)                  # the store's own level
        for t in range(QUARTERS):
            eff = GROWTH_APP * (t - g + 1) if t >= g else 0.0   # 0.8 in the first quarter, 1.6 in the second
            sales = 20 + own + 0.3 * t + shock[t] + eff + rng.normal(0, 0.8)
            rows.append((store, region, g, t, int(t >= g), shock[t], eff, sales))
stores = pd.DataFrame(rows, columns=["store", "region", "g", "t", "D",
                                     "region_shock", "true_eff", "sales"])

true_att_stores = stores.loc[stores["D"] == 1, "true_eff"].mean()
print(f"{stores['store'].nunique()} stores in {stores['region'].nunique()} regions, {QUARTERS} quarters")
print(f"true average effect on the treated store-quarters: {true_att_stores:.3f}")

**Your turn:** build `schools`: 9 districts of 6 schools over 10 years, with the start years in `district_start`. Columns `school`, `district`, `g`, `year`, `D`, `district_shock`, `true_eff`, `score`. Each district draws one shock per year from Normal(0, 0.5), shared by its schools; each school a level from Normal(0, 3); `score = 50 + level + 0.5 * year + district_shock + true_eff + Normal(0, 0.8)`, with `true_eff` growing by `GROWTH` points for each year of tutoring, the first year counting as 1.

In [ ]:
rng = np.random.default_rng(33)
NEVER = 10**9
district_start = {0: 4, 1: 4, 2: 4, 3: 6, 4: 6, 5: 8, 6: 8, 7: NEVER, 8: NEVER}   # first year of tutoring
SCHOOLS_PER_DISTRICT, YEARS = 6, 10
GROWTH = 1.5                                    # reading points per year of tutoring
# YOUR CODE HERE
schools = None

In [ ]:
assert schools is not None and len(schools) == 540, "540 rows: 9 districts x 6 schools x 10 years"
assert {"school", "district", "g", "year", "D", "district_shock", "true_eff", "score"} <= set(schools.columns), \
    "columns: school, district, g, year, D, district_shock, true_eff, score"
assert schools["school"].nunique() == 54, "54 different schools: give each school in each district its own name"
assert (schools.groupby("district")["g"].nunique() == 1).all(), "every school in a district starts in the same year"
assert (schools.groupby(["district", "year"])["district_shock"].nunique() == 1).all(), \
    "one shock per district and year, shared by all its schools: draw it outside the school loop"
assert schools.groupby("district")["district_shock"].nunique().min() == 10, "a new shock each year"
expected = np.where(schools["year"] >= schools["g"], GROWTH * (schools["year"] - schools["g"] + 1), 0.0)
assert np.allclose(schools["true_eff"], expected), "true_eff = GROWTH x (year - g + 1) from year g on, 0 before"
assert (schools["D"] == (schools["year"] >= schools["g"]).astype(int)).all(), "D is 1 from the start year on"
left = schools["score"] - 0.5 * schools["year"] - schools["district_shock"] - schools["true_eff"]
noise = left - left.groupby(schools["school"]).transform("mean")    # take out each school's own level
assert 0.6 < noise.std() < 1.0, "after the trend, shock, effect and level, what is left should be Normal(0, 0.8)"
true_att = schools.loc[schools["D"] == 1, "true_eff"].mean()
print(f"{(schools['D'] == 1).sum()} tutored school-years; true average effect on them {true_att:.3f}: keep it on file")

---
## 3. TWFE against the truth, and one forbidden comparison (PS3 Steps 2.1 and 2.2)

Two-way fixed effects is `y ~ D + C(unit) + C(period)`, the regression from Lab 11's Diagnosis 2. Goodman-Bacon showed it is a weighted average of 2×2s, and **every weight is positive**. The damage comes from the 2×2s whose control group is **already treated**: the later group's change minus the earlier group's change subtracts the earlier group's effect, still growing, as if it were trend. Compute one such 2×2 on group means. `df.groupby(["g", "t"])["y"].mean().unstack()` gives one row per start period, one column per period.

**Watch:** run this and read it.

In [ ]:
twfe_stores = smf.ols("sales ~ D + C(store) + C(t)", data=stores).fit()
print(f"TWFE {twfe_stores.params['D']:.3f} | true average effect on the treated {true_att_stores:.3f}")

# Forbidden 2x2: regions starting in quarter 7 (treated) against regions that started in quarter 3,
# which are already treated in every quarter of the window (the control)
cohort_means = stores.groupby(["g", "t"])["sales"].mean().unstack()
late, early = cohort_means.loc[7], cohort_means.loc[3]
before, after = list(range(3, 7)), list(range(7, 10))
forbidden_stores = ((late[after].mean() - late[before].mean())
                    - (early[after].mean() - early[before].mean()))
truth_late_stores = stores.loc[(stores["g"] == 7) & (stores["D"] == 1), "true_eff"].mean()
print(f"forbidden 2x2 (7 vs 3) {forbidden_stores:.3f} | true effect of the quarter-7 regions {truth_late_stores:.3f}")

**Your turn:** fit TWFE on `schools` as `twfe_schools`. Then compute the forbidden 2×2 `forbidden`: the districts starting in year 8 (treated) against the districts that started in year 4 (control), with years 4–7 as before and 8–9 as after. Store the true average effect of the year-8 districts in their treated years as `truth_late`.

In [ ]:
# YOUR CODE HERE
twfe_schools = forbidden = truth_late = None

In [ ]:
assert twfe_schools is not None and "D" in twfe_schools.params, "fit score ~ D + C(school) + C(year)"
assert len(twfe_schools.params) == 1 + 1 + 53 + 9, "one dummy per school and per year, less one of each"
assert twfe_schools.params["D"] < true_att, "TWFE should come out below the true average effect here"
win = schools[schools["g"].isin([4, 8]) & schools["year"].between(4, 9)].copy()
win["late"], win["after"] = (win["g"] == 8).astype(int), (win["year"] >= 8).astype(int)
same = smf.ols("score ~ late * after", data=win).fit().params["late:after"]   # the same 2x2, as a regression
assert forbidden is not None and abs(forbidden - same) < 1e-8, \
    "forbidden: (year-8 districts' change) minus (year-4 districts' change), years 4-7 before, 8-9 after"
assert abs(truth_late - 2.25) < 1e-9, "truth_late: mean true_eff of the year-8 districts in years 8 and 9"
assert forbidden < truth_late - 1, "the year-4 districts' growing effect should pull this 2x2 far below the truth"
print(f"TWFE {twfe_schools.params['D']:.3f} | truth {true_att:.3f}")
print(f"forbidden 2x2 {forbidden:.3f} | true effect of the year-8 districts {truth_late:.3f}")
print("Its weight in TWFE is positive. The estimate subtracts the year-4 districts' rise from years 4-7 to 8-9.")

---
## 4. One clean comparison (PS3 Step 3.1)

A clean cell compares group `g` in period `t` only with units **not yet treated** at `t` (`df["g"] > t`, which includes the never-treated), and measures both changes from the same baseline, `g - 1`, the last period before `g` starts. Lab 11's `clean_att()` runs this over every cell and averages them; here you write one cell by hand, so you can read every line of it. One cell rests on two periods and a few groups, so it is noisy: the *Watch* cell lands about a point from its truth. Its last lines do every clean cell the same way and average them: that is what steadies the estimate, and it is what `clean_att()` does for you.

**Watch:** run this and read it.

In [ ]:
g, t = 3, 5
treated = stores[stores["g"] == g]
controls = stores[stores["g"] > t]               # not yet treated in quarter 5: regions starting at 7, and never
base = g - 1                                      # quarter 2, the last before these regions adopt

d_tr = (treated.loc[treated["t"] == t, "sales"].mean()
        - treated.loc[treated["t"] == base, "sales"].mean())
d_co = (controls.loc[controls["t"] == t, "sales"].mean()
        - controls.loc[controls["t"] == base, "sales"].mean())
cell_stores = d_tr - d_co
truth_cell_stores = treated.loc[treated["t"] == t, "true_eff"].mean()
print(f"clean cell (g=3, t=5) {cell_stores:.3f} | its true effect {truth_cell_stores:.3f}")
print(f"controls: regions starting at {sorted(int(x) for x in controls['g'].unique())} (10**9 = never)")

# the other end: every clean cell the same way, averaged by its treated store-quarters
est_sum = true_sum = n_sum = 0
for g_ in sorted(x for x in stores["g"].unique() if x != NEVER):
    tr, b_ = stores[stores["g"] == g_], g_ - 1
    for t_ in range(g_, QUARTERS):
        co = stores[stores["g"] > t_]            # never-treated are always here, so no cell is empty
        est = ((tr.loc[tr["t"] == t_, "sales"].mean() - tr.loc[tr["t"] == b_, "sales"].mean())
               - (co.loc[co["t"] == t_, "sales"].mean() - co.loc[co["t"] == b_, "sales"].mean()))
        n = (tr["t"] == t_).sum()
        est_sum, true_sum, n_sum = est_sum + n * est, true_sum + n * tr.loc[tr["t"] == t_, "true_eff"].mean(), n_sum + n
print(f"all clean cells, averaged {est_sum / n_sum:.3f} | true average effect {true_sum / n_sum:.3f}")

**Your turn:** compute the clean cell for the districts that start in **year 4**, in **year 6**, as `cell_est`, against districts not yet tutoring in year 6, with year 3 as the baseline. Store the true effect of that cell as `cell_truth`.

In [ ]:
# YOUR CODE HERE
cell_est = cell_truth = None

In [ ]:
two = schools[((schools["g"] == 4) | (schools["g"] > 6)) & schools["year"].isin([3, 6])].copy()
two["tut"], two["after"] = (two["g"] == 4).astype(int), (two["year"] == 6).astype(int)
same = smf.ols("score ~ tut * after", data=two).fit().params["tut:after"]   # the same cell, as a regression
assert cell_est is not None and abs(cell_est - same) < 1e-8, \
    "cell_est: controls are districts with g > 6 (year 8 and never), both changes from year 3 to year 6"
assert abs(cell_truth - 4.5) < 1e-9, "cell_truth: mean true_eff of the year-4 districts in year 6"
assert abs(cell_est - cell_truth) < 2, "a clean cell should land near its true effect"
print(f"clean cell {cell_est:.3f} | truth {cell_truth:.3f}   (the forbidden 2x2 was {forbidden:.3f} against {truth_late:.3f})")
print("Neither control group in this cell has been tutored yet, so no effect is subtracted.")

---
## 5. An event study with a named omitted period (PS3 Step 3.2)

Relative time is `e = t - g`: periods since the start, negative before it. Make one 0/1 column for each value of `e` **except one**, the omitted (reference) period, here `e = -1`. Never-treated units have no `e`, so they get 0 in every column. Fit those columns plus unit and period fixed effects. Every coefficient is then the difference from the omitted period, which is why you name it: omit a different period and every coefficient moves. Flat estimates before 0 are consistent with parallel trends; they do not establish it, because the assumption is about the periods after the start, which no data show without treatment.

**Watch:** run this and read it.

In [ ]:
REF_APP = -1                                     # the omitted period: the last quarter before adoption
stores["e"] = np.where(stores["g"] < NEVER, stores["t"] - stores["g"], np.nan)   # never-adopting: no e
event_times = sorted(int(e) for e in stores["e"].dropna().unique())

col = {}                                          # event time -> its dummy's name (names cannot hold a minus)
for e in event_times:
    if e == REF_APP:
        continue                                  # no dummy: this period is the baseline for all the others
    col[e] = f"pre{-e}" if e < 0 else f"post{e}"
    stores[col[e]] = (stores["e"] == e).astype(int)   # NaN == e is False, so never-adopters get 0

es_fit_stores = smf.ols("sales ~ " + " + ".join(col.values()) + " + C(store) + C(t)", data=stores).fit()
es_stores = pd.Series({e: (0.0 if e == REF_APP else es_fit_stores.params[col[e]]) for e in event_times})
truth_path = pd.Series({e: GROWTH_APP * (e + 1) if e >= 0 else 0.0 for e in event_times})
print(pd.DataFrame({"estimate": es_stores, "truth": truth_path}).round(2).T.to_string())

**Your turn:** set `ref_period = -1` and build the same event study on `schools`, as `es_coef`: a Series indexed by event time, every event time in the panel included, with the omitted period stored as 0.

In [ ]:
# YOUR CODE HERE
ref_period = es_coef = None

In [ ]:
assert ref_period == -1, "name the omitted period: ref_period = -1"
assert es_coef is not None and sorted(es_coef.index) == list(range(-8, 6)), \
    "one entry per event time in the panel, -8 to 5, the omitted one included"
assert es_coef[ref_period] == 0, "the omitted period is the baseline: store it as 0"
assert es_coef[[-4, -3, -2]].abs().max() < 1.5, "pre-periods should sit near 0 in this panel"
post = np.array([es_coef[e] for e in range(4)])
assert np.abs(post - GROWTH * np.arange(1, 5)).max() < 2, \
    "e = 0 to 3 should track GROWTH x (e + 1): did each event time get its own dummy, and -1 none?"
print(es_coef.round(2).to_frame("estimate").T.to_string())
print("Near 0 before the start, rising after. The pre-periods are evidence; parallel trends after the start is an assumption.")

---
## 6. Three standard errors, and the level that matches the design (PS3 Step 3.3)

`.fit(cov_type="cluster", cov_kwds={"groups": df["col"]})` lets the errors of every row with the same label be correlated. The point estimate does not change; only the standard error does. Cluster at the level where treatment was **assigned** and where the shared shocks live: here, the group. Clustering by unit allows a unit's years to be correlated but treats units in one group as independent, though they share every shock. Few groups is its own problem: with under about ten clusters even the right level can mislead, which is what PS3 Phase 4 takes up. In PS3 these fits also go into your module; write them here as plain code first.

**Watch:** run this and read it.

In [ ]:
f_app = "sales ~ D + C(store) + C(t)"
fits_stores = {
    "default (iid)": smf.ols(f_app, data=stores).fit(),
    "clustered by store": smf.ols(f_app, data=stores).fit(cov_type="cluster",
                                                          cov_kwds={"groups": stores["store"]}),
    "clustered by region": smf.ols(f_app, data=stores).fit(cov_type="cluster",
                                                           cov_kwds={"groups": stores["region"]})}
print(f"{'':<22}{'estimate':>9}{'std err':>9}{'clusters':>10}")
for label, f in fits_stores.items():
    n_cl = getattr(f, "n_groups", "none")        # the default fit has no clusters
    print(f"{label:<22}{f.params['D']:>9.3f}{f.bse['D']:>9.3f}{n_cl:>10}")

**Your turn:** fit `score ~ D + C(school) + C(year)` on `schools` three ways: `fit_default`, `fit_school` (clustered by school) and `fit_district` (clustered by district).

In [ ]:
# YOUR CODE HERE
fit_default = fit_school = fit_district = None

In [ ]:
fits = {"default (iid)": fit_default, "clustered by school": fit_school, "clustered by district": fit_district}
assert all(f is not None and "D" in f.params for f in fits.values()), "three fits of score ~ D + C(school) + C(year)"
assert np.allclose([f.params["D"] for f in fits.values()], twfe_schools.params["D"]), \
    "clustering changes the standard error, never the estimate"
assert fit_school.cov_type == "cluster" and fit_school.n_groups == 54, "fit_school: one cluster per school, 54"
assert fit_district.cov_type == "cluster" and fit_district.n_groups == 9, "fit_district: one cluster per district, 9"
assert fit_district.bse["D"] > fit_school.bse["D"], "the shared district shocks should make the district SE the largest"
for label, f in fits.items():
    print(f"{label:<24}{f.params['D']:>9.3f}{f.bse['D']:>9.3f}")
print("Tutoring was switched on by district, and the shocks are shared by district. Which SE matches that, and why are 9 clusters few?")

---
**Done.** A wide table made long with the 2×2 read off a coefficient, a nested panel with the truth on file, TWFE against that truth and one forbidden comparison, one clean comparison, an event study with its omitted period named, and three standard errors: the building blocks of PS3 Phases 1–3. Now open `ps3-starter.ipynb`.